# 01 - Data loading

Run this notebook once after cloning the repository. It downloads all three project datasets directly into the repository's `data/` directory:

- `data/sroie/`
- `data/cord_v2/`
- `data/synthetic_resume/`

The `data/` directory is ignored by Git, so the downloaded files will not be uploaded to GitHub. Rerunning the notebook reuses completed downloads.

## 1. Install packages

In [1]:
%pip install -q "datasets>=3.0,<5" "requests>=2.31,<3" "tqdm>=4.66,<5"

Note: you may need to restart the kernel to use updated packages.


## 2. Set paths and source versions

The revisions are fixed so every team member downloads the same data.

In [1]:
from pathlib import Path
import shutil
import zipfile

import requests
from datasets import load_dataset, load_from_disk
from tqdm.auto import tqdm


def is_repo_root(path: Path) -> bool:
    return (path / "README.md").exists() and (path / "notebooks").exists()


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current / "poisoned-paperwork"]
    for parent in current.parents:
        candidates.extend([parent, parent / "poisoned-paperwork"])

    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate

    raise RuntimeError("Could not find the local poisoned-paperwork repository.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
CACHE_DIR = DATA_DIR / ".cache"
DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

SROIE_ID = "jsdnrs/ICDAR2019-SROIE"
SROIE_REVISION = "bffe40c26759f3376ec2b3ae9031dbba54cd587c"
CORD_ID = "naver-clova-ix/cord-v2"
CORD_REVISION = "7f0115a4b758a71d6473b8d085751692da2fef98"
RESUME_REPO = "jijunhao/SyntheticResumeData"
RESUME_REVISION = "3ecd09129f65955e743aacf67f2785fcee10d461"

print("Repository:", REPO_ROOT)
print("Data directory:", DATA_DIR)

Repository: /Users/prabudhdkandpal/Desktop/CSCI566/code/poisoned-paperwork
Data directory: /Users/prabudhdkandpal/Desktop/CSCI566/code/poisoned-paperwork/data


## 3. Download SROIE

In [2]:
def load_or_download_huggingface(name: str, dataset_id: str, revision: str):
    destination = DATA_DIR / name
    if (destination / "dataset_dict.json").exists():
        print(f"Loading existing {name} data from {destination}")
        return load_from_disk(destination)

    if destination.exists() and any(destination.iterdir()):
        raise RuntimeError(
            f"{destination} contains an incomplete download. "
            "Remove that folder and run this cell again."
        )

    print(f"Downloading {dataset_id}...")
    dataset = load_dataset(
        dataset_id,
        revision=revision,
        cache_dir=str(CACHE_DIR / "huggingface"),
    )
    dataset.save_to_disk(destination)
    return dataset


sroie = load_or_download_huggingface("sroie", SROIE_ID, SROIE_REVISION)
print({split: len(data) for split, data in sroie.items()})

README.md:   0%|          | 0.00/14.3k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  319MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/626 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/361 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/626 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/361 [00:00<?, ? examples/s]

{'train': 626, 'test': 361}


## 4. Download CORD v2

In [3]:
cord = load_or_download_huggingface("cord_v2", CORD_ID, CORD_REVISION)
print({split: len(data) for split, data in cord.items()})

README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

data/train-00000-of-00004-b4aaeceff1d90e(…): reconstructing file:   0%|          |  0.00B /  490MB            

data/train-00000-of-00004-b4aaeceff1d90e(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-7dbbe248962764(…): reconstructing file:   0%|          |  0.00B /  441MB            

...

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

{'train': 800, 'validation': 100, 'test': 100}


## 5. Download SyntheticResumeData

In [4]:
def download_file(url: str, destination: Path) -> None:
    if destination.exists():
        print(f"Using existing archive: {destination.name}")
        return

    partial = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with partial.open("wb") as output, tqdm(
            total=total, unit="B", unit_scale=True, desc=destination.name
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
                    progress.update(len(chunk))
    partial.replace(destination)


def extract_github_archive(archive: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for member in tqdm(bundle.infolist(), desc="Extracting resumes"):
            parts = Path(member.filename).parts
            if len(parts) < 2:
                continue
            relative = Path(*parts[1:])
            if ".." in relative.parts:
                raise ValueError(f"Unsafe archive path: {member.filename}")
            target = destination / relative
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(member) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)


resume_dir = DATA_DIR / "synthetic_resume"
resume_pdf_dir = resume_dir / "resume_list_pdf"
resume_annotation_dir = resume_dir / "result_gt"
resume_pdfs = list(resume_pdf_dir.glob("*.pdf"))
resume_annotations = list(resume_annotation_dir.glob("*.json"))

if len(resume_pdfs) != 2994 or len(resume_annotations) != 2994:
    archive = CACHE_DIR / f"SyntheticResumeData-{RESUME_REVISION}.zip"
    url = f"https://github.com/{RESUME_REPO}/archive/{RESUME_REVISION}.zip"
    download_file(url, archive)
    extract_github_archive(archive, resume_dir)
    resume_pdfs = list(resume_pdf_dir.glob("*.pdf"))
    resume_annotations = list(resume_annotation_dir.glob("*.json"))
else:
    print("Loading existing SyntheticResumeData files")

print(f"PDFs: {len(resume_pdfs):,}")
print(f"Annotations: {len(resume_annotations):,}")

SyntheticResumeData-3ecd09129f65955e743aacf67f2785fcee10d461.zip: 0.00B [00:00, ?B/s]

Extracting resumes:   0%|          | 0/5995 [00:00<?, ?it/s]

PDFs: 2,994
Annotations: 2,994


## 6. Verify the downloads

In [5]:
counts = {
    "SROIE documents": sum(len(split) for split in sroie.values()),
    "CORD v2 documents": sum(len(split) for split in cord.values()),
    "Synthetic resume PDFs": len(resume_pdfs),
    "Synthetic resume annotations": len(resume_annotations),
}
expected = {
    "SROIE documents": 987,
    "CORD v2 documents": 1000,
    "Synthetic resume PDFs": 2994,
    "Synthetic resume annotations": 2994,
}

for name, count in counts.items():
    print(f"{name}: {count:,}")

assert counts == expected, f"Expected {expected}, but found {counts}"
print("\nAll datasets were downloaded successfully.")

SROIE documents: 987
CORD v2 documents: 1,000
Synthetic resume PDFs: 2,994
Synthetic resume annotations: 2,994

All datasets were downloaded successfully.


Next, run `02_data_exploration.ipynb` to view example documents and basic dataset statistics.